# 02: Do the flaws matter for a model?

Notebook 01 confirmed the flaws in the data. This notebook asks whether they change what a model learns. There are four small experiments, each tied to one claim in the report.

**Every experiment reports the same two numbers:**
- **Attacks caught:** the share of attack rows the model flags.
- **False alarms:** the share of normal rows the model flags.

**Same model everywhere:** scikit-learn's `HistGradientBoostingClassifier` with default settings, threshold 0.5. The point is to compare data, not to tune a model.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

from cybershield.ml.train_zeek import fit_zeek_model
from cybershield.ml.zeek_features import (
    DATASETS_DIR, ZEEK_CAT_COLS, ZEEK_FLOW_RATIO_COLS, ZEEK_RAW_NUM_COLS, load_cic2017_zeek, load_unsw_official_zeek,
)

SEED = 42


def score(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    return {
        "Attacks caught": f"{(pred[y == 1] == 1).mean():.1%}" if (y == 1).any() else "-",
        "False alarms": f"{(pred[y == 0] == 1).mean():.1%}" if (y == 0).any() else "-",
    }


# UNSW-NB15 official training and testing CSVs
train = pd.read_csv(DATASETS_DIR / "unsw_nb15/UNSW_NB15_training-set.csv")
test = pd.read_csv(DATASETS_DIR / "unsw_nb15/UNSW_NB15_testing-set.csv")
features = [c for c in train.columns if c not in ("id", "label", "attack_cat")]
for col in ["proto", "service", "state"]:
    cats = sorted(set(train[col]) | set(test[col]))
    train[col], test[col] = pd.Categorical(train[col], categories=cats), pd.Categorical(test[col], categories=cats)


def fit_csv(df, cols):
    return HistGradientBoostingClassifier(random_state=SEED, categorical_features="from_dtype").fit(df[cols], df.label)


# Zeek conn.log fields used for the Zeek models: 11 raw fields + 3 simple ratios (no IPs, ports or timestamps)
ZEEK_FIELDS = ZEEK_CAT_COLS + ZEEK_RAW_NUM_COLS + ZEEK_FLOW_RATIO_COLS

## Experiment 1: Can TTL alone detect attacks?

**Claim:** TTL gives away the label. Sarhan et al. found one TTL feature is almost as accurate as all the features combined.

**Test:** train on the official training CSV, then score the official testing CSV with three models: all features, TTL only (`sttl`), and all features except the three TTL fields.

In [2]:
ttl = ["sttl", "dttl", "ct_state_ttl"]
setups = {"All features": features, "TTL only (sttl)": ["sttl"], "All features except TTL": [c for c in features if c not in ttl]}
pd.DataFrame({name: score(test.label, fit_csv(train, cols).predict(test[cols])) for name, cols in setups.items()}).T

,Attacks caught,False alarms
All features,98.6%,26.2%
TTL only (sttl),99.9%,51.8%
All features except TTL,98.2%,26.4%


**Verdict: partly confirmed.**
- TTL alone catches almost every attack: as many as all features combined.
- But on its own it also flags about half of the normal traffic, twice as many false alarms as the full model.
- Removing the three TTL fields barely changes the full model. The other features carry the same information, so dropping TTL doesn't remove the problem.
- So TTL is a strong shortcut for finding attacks, but it doesn't separate normal traffic well by itself.

**Also visible:** even the full model flags about a quarter of the testing CSV's normal traffic. That fits Notebook 01, A3: the testing CSV's normal traffic differs from the training CSV's.

## Experiment 2: Do duplicates inflate scores?

**Claim:** 42% of training rows and 34.5% of testing rows are duplicates (Notebook 01, A2). Duplicates can make a model look better than it is.

**Test:** score the same model on the testing CSV with and without its duplicate rows, and train it with and without the training CSV's duplicates.

In [3]:
train_dedup = train.drop_duplicates(features + ["label"])
test_dedup = test.drop_duplicates(features + ["label"])
runs = {
    "Train as published, test as published": (train, test),
    "Train as published, test without duplicates": (train, test_dedup),
    "Train without duplicates, test without duplicates": (train_dedup, test_dedup),
}
pd.DataFrame({name: score(te.label, fit_csv(tr, features).predict(te[features])) for name, (tr, te) in runs.items()}).T

,Attacks caught,False alarms
"Train as published, test as published",98.6%,26.2%
"Train as published, test without duplicates",97.1%,25.3%
"Train without duplicates, test without duplicates",97.3%,24.9%


**Verdict: small effect.** Removing duplicates lowers attacks caught by only a point or two, and barely changes false alarms. The duplicates are a data-quality problem, but here they don't distort the scores much.

## Experiment 3: Is there a shortcut in UNSW-NB15's Zeek version?

**Claim:** Converting to Zeek removes TTL, but `conn_state` (how a connection ended) becomes a new shortcut, because unanswered connections are far more common among attacks.

**Test:** using the creators' Zeek logs, train on Day 2 and score Day 1, a different day. Compare a model with all Zeek fields against models that each get a single field. If one field alone matched the full model, as TTL did in experiment 1, it would be a shortcut.

In [4]:
unsw = load_unsw_official_zeek()
day2, day1 = unsw[unsw.day == "D2"], unsw[unsw.day == "D1"]


def zeek_score(train_df, test_df, cols):
    model = fit_zeek_model(train_df, cols)
    return score(test_df.label, (model.predict_proba(test_df[cols])[:, 1] >= 0.5).astype(int))


results = {"All Zeek fields": zeek_score(day2, day1, ZEEK_FIELDS)}
for field in ["conn_state", "duration", "orig_bytes", "orig_pkts", "resp_bytes", "service"]:
    results[f"Only {field}"] = zeek_score(day2, day1, [field])
pd.DataFrame(results).T

,Attacks caught,False alarms
All Zeek fields,76.8%,0.2%
Only conn_state,17.1%,0.1%
Only duration,10.8%,0.1%
Only orig_bytes,37.9%,0.2%
Only orig_pkts,44.9%,0.7%
Only resp_bytes,9.8%,0.1%
Only service,2.8%,0.0%


**Verdict: the claim is overstated.**
- `conn_state` alone catches under a fifth of attacks: roughly the attacks that went unanswered.
- No single field comes close to the full model, unlike TTL in experiment 1.
- So unanswered connections are a useful signal here, but not a TTL-like shortcut. **The reports should be corrected.**

## Experiment 4: Does a model trained on UNSW-NB15 work on another network?

**Claim:** Results on UNSW-NB15 won't carry over to other data (Option A's main weakness).

**Test:** train one model on UNSW-NB15's Zeek logs and another on CIC-IDS2017 Tuesday to Thursday. Score both on two CIC-IDS2017 days neither saw:
- **Monday:** only normal traffic, so it measures false alarms.
- **Friday:** attacks neither model was trained on (bot, port scan, DDoS).

*CIC-IDS2017 here is the HuggingFace mirror, labelled from UNB's attack schedule.*

In [5]:
cic = load_cic2017_zeek()
models = {
    "Trained on UNSW-NB15": fit_zeek_model(unsw, ZEEK_FIELDS),
    "Trained on CIC-IDS2017 (Tue-Thu)": fit_zeek_model(cic[cic.day.isin(["tuesday", "wednesday", "thursday"])], ZEEK_FIELDS),
}
rows = {}
for name, model in models.items():
    row = {}
    for day in ["monday", "friday"]:
        d = cic[cic.day == day]
        s = score(d.label, (model.predict_proba(d[ZEEK_FIELDS])[:, 1] >= 0.5).astype(int))
        if day == "monday":
            row["Monday false alarms"] = s["False alarms"]
        else:
            row["Friday attacks caught"], row["Friday false alarms"] = s["Attacks caught"], s["False alarms"]
    rows[name] = row
pd.DataFrame(rows).T

,Monday false alarms,Friday attacks caught,Friday false alarms
Trained on UNSW-NB15,48.3%,1.0%,45.8%
Trained on CIC-IDS2017 (Tue-Thu),0.4%,73.2%,0.1%


**Verdict: confirmed, strongly.**
- The UNSW-trained model flags about half of CIC-IDS2017's normal traffic and catches almost none of its attacks.
- The CIC-trained model raises few false alarms and catches most of Friday's attacks, even though it never saw those attack types.
- A model that learns UNSW-NB15's normal traffic doesn't recognise normal traffic from another network.

## Summary

| # | Question | Answer |
|---|---|---|
| 1 | Can TTL alone detect attacks? | It catches nearly all attacks, but with twice the false alarms |
| 2 | Do duplicates inflate scores? | Only slightly |
| 3 | Is `conn_state` a TTL-like shortcut in the Zeek version? | No: a useful signal, not a shortcut. The reports overstate it |
| 4 | Does a UNSW-trained model work on another network? | No: about half of normal traffic flagged, almost no attacks caught |